# logit-thermo A2-23 第 1 段：(A) 割る型の罰の予言のための読み直しと、(D) 指示調整版 2 本の融点さがし——ランタイム 1/3（GPU、L4）

- **必要な secret：なし**（`HF_TOKEN` があれば取り込みが少し速くなるだけ。どのモデルも許可なしで落とせます）。
- **ランタイム：L4**（ハイメモリはいりません）。GPU のメモリが 20GB に足りなければ、⚠ を送ってランタイムを切ります。
- **3 本のノートブック（ランタイム 1/3・2/3・3/3）を、別々のランタイムで同時に回してください。** 仕事は 30 個：
  - (D) 指示調整版 2 本（Qwen2.5-1.5B-Instruct・SmolLM2-1.7B-Instruct）の融点さがし（罰なし、温度 15 個、記事 40 本）。1 つ 1 時間くらい。
  - (A) 小さい 7 本の罰なしの文（G-pre と A2-19 の対照）を、生成せずに読み直す仕事 28 個。1 つ数分〜10 分くらい。
  - 3 つ同時なら 1〜2 時間、CU は合わせて 8 くらいの見込みです。
- **データは、予言のコードを回すまで開けません。** 結果も記事の中身も表示しません。
- 計算のコード：これまでのデータ集めのコードは Drive の前のノートブックから取り出してハッシュで確かめる。`scripts/a2_18_collect.py`（`0b9d203ebe0a71e6…`）・`a2_19_collect.py`（`de4d86520b5ab6a4…`）・`a2_20_collect.py`（`29d5dbd7a0562043…`）・`a2_23_collect.py`（`f881ad8ebeaeafd4…`）は、Drive の `A2_explore/` の .py から読んでハッシュで確かめる。
- 読み直しは Drive の `A2_gpre/` と `A2_19/` の npz を読みます（書き換えません）。保存先 `A2_23_s1/`。途中で切れても、もう一度「すべて実行」すれば、終わった仕事は飛ばします。
- **通知**：🚀・仕事ごとの ✅・🎉・⚠️。名札は「logit-thermo A2-23 第1段（割る型の罰の読み直し 28 個・指示調整版 2 本の融点さがし、ランタイム 1/3、L4）」。終わったら、またはどのセルでもエラーで止まったら、ランタイムを切ります（手で止めたときも切れます）。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo A2-23 第1段（割る型の罰の読み直し 28 個・指示調整版 2 本の融点さがし、ランタイム 1/3、L4）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - A2-23 stage1 rescoring and instruct melting scan worker 1of3', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import torch
if not torch.cuda.is_available() or torch.cuda.get_device_properties(0).total_memory < 20e9:
    name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU なし'
    notify(f'⚠️ GPU のメモリが足りません（{name}）。L4 を選んでください。ランタイムを切ります', tags='warning')
    from google.colab import runtime; runtime.unassign()
    raise SystemExit
torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass
def _cut_on_error(result):
    # どのセルでエラーになっても（A2-19 ではセル 2 で止まったときに切れなかった）、⚠ を送ったあとでランタイムを切る。手で止めたときも切れます
    if result.error_in_exec is not None and globals().get('AUTO_DISCONNECT', True):
        time.sleep(10)
        try:
            from google.colab import drive as _d; _d.flush_and_unmount()
        except Exception:
            pass
        from google.colab import runtime as _r; _r.unassign()
if not globals().get('_CUT_ON'):
    get_ipython().events.register('post_run_cell', _cut_on_error); _CUT_ON = True

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：計算のコード
import ast, glob, hashlib, json
def take_src(want):
    """want：{変数の名前: sha256}。Drive の logit-thermo のノートブックから、埋め込んだコードを取り出してハッシュで確かめる。"""
    got = {}
    for p in sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/*.ipynb', recursive=True)):
        try:
            cells = json.load(open(p))['cells']
        except Exception:
            continue
        for c in cells:
            s = c['source'] if isinstance(c['source'], str) else ''.join(c['source'])
            if c.get('cell_type') != 'code' or 'SRC' not in s:
                continue
            try:
                body = ast.parse(s).body
            except SyntaxError:
                continue
            for n in body:
                if isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name) and n.targets[0].id in want and n.targets[0].id not in got:
                    v = ast.literal_eval(n.value)
                    if hashlib.sha256(v.encode()).hexdigest() == want[n.targets[0].id]:
                        got[n.targets[0].id] = v
        if len(got) == len(want):
            break
    miss = [k for k in want if k not in got]
    assert not miss, ('Drive のノートブックからコードを取り出せませんでした（前のノートブックを動かしていませんか）：', miss)
    return got
_s = take_src({'SRC22': 'e578cfc480980be6b0d1e18377b6beb8b7555e0744f56b13f63cb304a9f73d53', 'SRC24': '6ef87866ae7d895d63c6e07d48a58a2ddf4111a4802ffe98dbe7f61892c96b57', 'SRC7C': '8f8776cbe4a0e1a7b1f676c4f84ba70fb9c6c0b46ef1466969e09f0ba062b550', 'SRC7BC': 'b03ff9e2b16e2297fb9142a26e15f561caacf77216453d78e2024066c01c57d0', 'SRCF': '0ac42c7ab82b06d2e7443ba9c0c0e147c0121b5351e0b5830b1a54b46043190d', 'SRCD': '33f47a0a3bbda18341205dedb94c6d200f295fc4136f5cb8e0ee067b2d7f518c', 'SRCD2': 'cddae786903604183e46708a6606b482f720bba0b0df0a8df724440c9bde51b2', 'SRCG': '49aa63e0c3479f9841bf4b17740325fe21ddaea8b118f03730461959ef03e44b'})
SRC22, SRC24, SRC7C, SRC7BC, SRCF, SRCD = _s['SRC22'], _s['SRC24'], _s['SRC7C'], _s['SRC7BC'], _s['SRCF'], _s['SRCD']
SRCD, SRCD2 = _s['SRCD'], _s['SRCD2']   # 名前を 2 つ並べて置く（take_src が読み飛ばす形）
SRCD2, SRCG = _s['SRCD2'], _s['SRCG']   # 名前を 2 つ並べて置く（take_src が読み飛ばす形）
def take_file(name, want):
    """Drive の logit-thermo のフォルダから、名前とハッシュが合う .py のファイルを読む。"""
    for p in sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/' + name, recursive=True)):
        s = open(p, encoding='utf-8').read()
        if hashlib.sha256(s.encode()).hexdigest() == want:
            return s
    raise AssertionError(('Drive に ' + name + ' （ハッシュが合うもの）が見つかりません', want))
SRC17 = take_file('a2_18_collect.py', '0b9d203ebe0a71e67d3c419897212c2091f04697c0187ac9475f9112eeea6496')
SRC19 = take_file('a2_19_collect.py', 'de4d86520b5ab6a42f02a7dea8a3048a943711ba26499d3a08ca061aad27b3ef')
SRC20 = take_file('a2_20_collect.py', '29d5dbd7a05620433f5bf0d1ad9dc3c9e130f89d8ed66622139149d51fea99d6')
SRC23 = take_file('a2_23_collect.py', 'f881ad8ebeaeafd43f8089fa717b4f493a620eaf8c520866329543ccde9b2a74')
exec(SRC22)
exec(SRC24)
exec(SRC7C)
exec(SRC7BC)
exec(SRCF)
exec(SRCD)
exec(SRCD2)
exec(SRCG)
exec(SRC17)
exec(SRC19)
exec(SRC20)
exec(SRC23)
print('コードを置きました')


In [ ]:
# セル3：自己テスト → BBC の記事（A2-2・A2-19 と同じか目録で確かめる）→ 仕事を取りに行って回す → 保存（結果も記事も表示しない）
import json, sys, transformers
try:
    print('自己テスト：', self_test_a27('cuda'), self_test_7b('cuda'), self_test_field('cuda'), self_test_dsc('cuda'), self_test_dsc2('cuda'), self_test_gpre(), self_test_a18(), self_test_a19(), self_test_a20(), self_test_a23('cuda'), self_test_chat())
    os.makedirs('/content/drive/MyDrive/logit-thermo/A2_23_s1', exist_ok=True)
    assert SEED == 20261027, ('記事を選ぶ種が A2-2 と違います', SEED)
    docs, meta = load_docs()
    check_docs20(meta, json.load(open('/content/drive/MyDrive/logit-thermo' + '/A2_2/docs_meta.json')), 'docs')
    GROUPS, META19 = load_docs19('/content/drive/MyDrive/logit-thermo' + '/A2_2/docs_meta.json')
    check_docs20(META19, json.load(open('/content/drive/MyDrive/logit-thermo' + '/A2_19/docs_meta_w0.json')), 'groups')
    JOBS = a23_jobs1()
    docs_of = lambda j: GROUPS[j[5]] if j[2] == 'rs_c0' else docs
    srcdir_of = lambda j: ('/content/drive/MyDrive/logit-thermo' + '/A2_gpre') if j[2] == 'rs_gpre' else ('/content/drive/MyDrive/logit-thermo' + '/A2_19') if j[2] == 'rs_c0' else None
    N_DONE, LEFT = run_a23(JOBS, 0, 3, docs_of, srcdir_of, '/content/drive/MyDrive/logit-thermo/A2_23_s1', 'cuda', 20, notify=lambda s: notify(s, tags='warning') if s.startswith('⚠') else notify('✅ ' + s, tags='white_check_mark'))
    json.dump(dict(worker=0, done=N_DONE, left=LEFT, samples=GP_S, chunk=20, a23_sha256='f881ad8ebeaeafd43f8089fa717b4f493a620eaf8c520866329543ccde9b2a74', a20_sha256='29d5dbd7a05620433f5bf0d1ad9dc3c9e130f89d8ed66622139149d51fea99d6', a19_sha256='de4d86520b5ab6a42f02a7dea8a3048a943711ba26499d3a08ca061aad27b3ef',
                   torch=torch.__version__, transformers=transformers.__version__, python=sys.version.split()[0], gpu=torch.cuda.get_device_name(0),
                   matmul=torch.get_float32_matmul_precision()),
              open('/content/drive/MyDrive/logit-thermo/A2_23_s1' + '/run_meta_worker0.json', 'w'), ensure_ascii=False, indent=1)
    print('保存しました')
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ A2-23 第1段（ランタイム 1/3）で止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify(f'🎉 終わりました：A2-23 第1段（ランタイム 1/3）。このランタイムで {N_DONE} 個。まだ残っている仕事：{len(LEFT)} 個' + (f'（{", ".join(LEFT)}）' if LEFT else '（全部そろいました）'), tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
